[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ersilia-os/ub-cedd-projects-workshop/blob/main/sandbox/gnina_docking_screen.ipynb)

# gnina docking screen

**Sandbox notebook - not workshop material.**

The question this notebook answers is: *can we upload a receptor-ligand complex to Colab, reproduce
the ligand's pose with docking, and then screen a small library in the same pocket?* It follows the
authors' own [gnina Colab](https://colab.research.google.com/drive/1QYo5QLUE80N_G28PlpYs6OKGddhhd931)
(download the binary, split a PDB with `grep`, redock with `--autobox_ligand`, view with py3Dmol),
with the test case swapped for an uploaded complex and a library screen added on the end.

It was written with the blue group's CpABC1-silymarin complex in mind
(`Projects/BlueTeam/Data/CpABC1-Silymarin.pdb` in the shared Drive), but any PDB file holding a
protein with one bound molecule works.

gnina is a docking program that reports the classical Vina affinity plus two scores from a
convolutional neural network ([Ragoza et al., J. Chem. Inf. Model. 2017](https://doi.org/10.1021/acs.jcim.6b00740),
[repository](https://github.com/gnina/gnina), Apache 2.0).

## What you will do

- Download the gnina binary and get it running on today's Colab image
- Upload a receptor-ligand complex and split it into its two parts
- Redock the reference ligand and measure how far the pose moved
- Dock a small library of silymarin analogues into the same pocket
- Rank the compounds and look at the best one

Set the runtime to **T4 GPU** (*Runtime > Change runtime type*) before you start. The whole notebook
takes about 40 minutes, almost all of it in the library screen.

## 1. Get gnina

The release binary is 1.4 GB and needs no installation. The Python packages do: Colab ships none of
rdkit, py3Dmol or stylia.

The binary was built against CUDA 12 and Colab has moved to CUDA 13. Most CUDA 12 libraries are
still in the image, as leftovers of the older PyTorch wheels, but `libnvToolsExt.so.1` is not, so we
put it in a folder of its own and point `LD_LIBRARY_PATH` at everything CUDA 12 we can find. Nothing
in the Python environment is touched.

> **Note:** if gnina reports `error while loading shared libraries: libsomething.so.12`, Colab has
> dropped another one. Add its `nvidia-<something>-cu12` wheel to the same `--target` install.

In [ ]:
!pip install -q rdkit py3Dmol stylia

Download the binary, fetch the missing CUDA library and check that gnina starts. The version line
is the thing to report if anything below fails.

In [ ]:
import glob, os, stat, subprocess, sys, urllib.request

WORK = "work"  # everything this notebook produces goes here
os.makedirs(WORK, exist_ok=True)
GNINA = f"{WORK}/gnina"

if not os.path.exists(GNINA):
    urllib.request.urlretrieve("https://github.com/gnina/gnina/releases/download/v1.3.2/gnina.1.3.2", GNINA)
    os.chmod(GNINA, os.stat(GNINA).st_mode | stat.S_IEXEC)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--target", f"{WORK}/cuda12",
                    "nvidia-nvtx-cu12==12.1.105"], check=True)

libs = glob.glob("/usr/local/lib/python3.*/dist-packages/nvidia/*/lib/*.so*") + glob.glob(f"{WORK}/cuda12/nvidia/*/lib/*.so*")
ENV = {**os.environ, "LD_LIBRARY_PATH": ":".join(sorted({os.path.abspath(os.path.dirname(p)) for p in libs}))}
print("Python", sys.version.split()[0], "|", subprocess.run([GNINA, "--version"], capture_output=True, text=True, env=ENV).stdout.strip())
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip() or "NO GPU: set the runtime to T4")

## 2. Upload the complex

Docking needs a protein (the *receptor*) and a molecule already sitting in the pocket (the
*reference ligand*), which is what says where to search. Both come from one uploaded PDB file.

Run the cell, click **Choose Files** and pick the complex. The file stays in this Colab session only.
`files.upload()` needs a human, so this is the one cell that cannot run unattended.

In [ ]:
from google.colab import files

uploaded = files.upload()
name = next(iter(uploaded))
complex_path = f"{WORK}/{name}"
with open(complex_path, "wb") as f:
    f.write(uploaded[name])
print(f"{complex_path} | {os.path.getsize(complex_path) / 1e6:.1f} MB")

## 3. Split the complex

`ATOM` lines are the protein, `HETATM` lines are everything else: the bound molecule, but also water
and ions. The largest group of `HETATM` lines sharing one residue is taken as the ligand.

In [ ]:
lines = open(complex_path).read().splitlines()
protein = [line for line in lines if line.startswith("ATOM")]

residues = {}
for line in lines:
    if line.startswith("HETATM") and line[17:20].strip() not in ("HOH", "WAT"):
        residues.setdefault(line[17:26], []).append(line)  # residue name, chain and number

residue, ligand_lines = max(residues.items(), key=lambda item: len(item[1]))
with open(f"{WORK}/receptor.pdb", "w") as f:
    f.write("\n".join(protein) + "\nEND\n")
print(f"receptor: {len(protein)} atoms | ligand ({residue.strip()}): {len(ligand_lines)} atoms")

A PDB file records atom positions but not bonds, so RDKit works them out from the distances. The
SMILES string is the quick check that the right thing came out: for the CpABC1 complex it should be
silybin, the main component of silymarin.

In [ ]:
from rdkit import Chem
from rdkit.Chem import rdDetermineBonds

ligand = Chem.MolFromPDBBlock("\n".join(ligand_lines), removeHs=False)
rdDetermineBonds.DetermineBondOrders(ligand, charge=0)
Chem.MolToMolFile(ligand, f"{WORK}/ligand.sdf")
print(Chem.MolToSmiles(Chem.RemoveHs(ligand)))

## 4. Redock the reference ligand

Take the ligand out, search for its position again, and see whether it comes back. `--autobox_ligand`
draws the search box around the reference ligand, `--seed 0` fixes the random numbers.

`EXHAUSTIVENESS` is how hard the search works. On a T4 with two CPU cores, 8 takes about 210 seconds
for a molecule the size of silybin; 4 takes 140 and found the same best pose here. The search itself
runs on the CPU, which is why the GPU does not make this fast.

In [ ]:
EXHAUSTIVENESS = 8

def dock(ligand_file, tag):
    """Dock one ligand into the receptor, inside the box around the reference ligand."""
    out_file = f"{WORK}/{tag}_docked.sdf"
    result = subprocess.run(
        [GNINA, "-r", f"{WORK}/receptor.pdb", "-l", ligand_file, "--autobox_ligand", f"{WORK}/ligand.sdf",
         "-o", out_file, "--seed", "0", "--exhaustiveness", str(EXHAUSTIVENESS)],
        capture_output=True, text=True, env=ENV)
    if result.returncode != 0:
        raise RuntimeError(result.stderr[-2000:])
    return out_file

dock(f"{WORK}/ligand.sdf", "reference")

Each pose carries three scores: `affinity` is the Vina binding energy in kcal/mol (more negative is
better), `cnn_pose_score` is the network's confidence that the pose is right (0 to 1), and
`cnn_affinity` is its binding-strength estimate as a pK value (higher is better). The poses come out
ordered by `cnn_pose_score`.

In [ ]:
import pandas as pd
from rdkit import RDLogger

RDLogger.DisableLog("rdApp.warning")  # gnina's files are fine, RDKit just complains about them

def read_scores(docked_file):
    """Read the scores gnina attached to each pose in an output file."""
    rows = []
    for mol in Chem.SDMolSupplier(docked_file):
        if mol is not None:
            rows.append({"name": mol.GetProp("_Name") or "ligand",
                         "affinity": float(mol.GetProp("minimizedAffinity")),
                         "cnn_pose_score": float(mol.GetProp("CNNscore")),
                         "cnn_affinity": float(mol.GetProp("CNNaffinity"))})
    return pd.DataFrame(rows)

read_scores(f"{WORK}/reference_docked.sdf")

## 5. Check the redocked pose

RMSD is the average distance between matching atoms of two poses, in angstroms; below 2 they count as
the same pose. Worth measuring for both candidates: the pose the network ranks first, and the pose
with the best Vina affinity.

In [ ]:
from rdkit.Chem import rdMolAlign

def best_pose(docked_file, by="affinity"):
    """Return the best pose in a gnina output file, by Vina affinity or by CNN pose score."""
    poses = [mol for mol in Chem.SDMolSupplier(docked_file) if mol is not None]
    if by == "affinity":
        return min(poses, key=lambda mol: float(mol.GetProp("minimizedAffinity")))
    return max(poses, key=lambda mol: float(mol.GetProp("CNNscore")))

reference = Chem.MolFromMolFile(f"{WORK}/ligand.sdf")
for by in ("affinity", "cnn_pose_score"):
    rmsd = rdMolAlign.CalcRMS(best_pose(f"{WORK}/reference_docked.sdf", by), reference)
    print(f"best pose by {by:<15} RMSD {rmsd:5.2f} angstrom")
Chem.MolToMolFile(best_pose(f"{WORK}/reference_docked.sdf"), f"{WORK}/reference_best.sdf")

On the CpABC1 complex the Vina-best pose comes back at 0.50 angstrom while the network's favourite
is 10.6 away, so the search works but the CNN ranking does not hold up on this receptor. That is
what you would expect from a network trained on crystal structures, applied to a computed model.
**The ranking below therefore uses the Vina affinity**, with the CNN scores kept as extra columns.

This is the check worth doing on any new target before trusting a score.

Look at the two poses. Drawing every protein atom is slow, so only the residues lining the pocket
are kept, as thin sticks. The uploaded pose is grey, the redocked one green.

In [ ]:
import numpy as np

centre = reference.GetConformer().GetPositions().mean(axis=0)
coords = np.array([[float(line[30:38]), float(line[38:46]), float(line[46:54])] for line in protein])
pocket = [line for line, near in zip(protein, np.linalg.norm(coords - centre, axis=1) < 12) if near]
print(f"{len(pocket)} atoms within 12 angstrom of the ligand")

The viewer is interactive: drag to rotate, scroll to zoom.

In [ ]:
import py3Dmol

def view_pose(pose_file, reference_file=None):
    """Show a docked pose (green) in the pocket, next to a reference pose (grey)."""
    view = py3Dmol.view(width=700, height=450)
    view.addModel("\n".join(pocket), "pdb")
    view.setStyle({"stick": {"colorscheme": "whiteCarbon", "radius": 0.08}})
    if reference_file:
        view.addModel(open(reference_file).read(), "sdf")
        view.setStyle({"model": 1}, {"stick": {"colorscheme": "greyCarbon"}})
    pose_model = 2 if reference_file else 1
    view.addModel(open(pose_file).read(), "sdf")
    view.setStyle({"model": pose_model}, {"stick": {"colorscheme": "greenCarbon"}})
    view.zoomTo({"model": pose_model})
    return view

view_pose(f"{WORK}/reference_best.sdf", f"{WORK}/ligand.sdf")

## 6. Build the compound library

`silymarin_analogues.csv` next to this notebook holds sixteen public compounds related to silymarin:
the other flavonolignans of the extract and simpler flavonoids sharing the same core, with their
PubChem CIDs. The first ten are a deliberate mix of large and small, and those are the ones screened
by default. Any CSV with `name` and `smiles` columns works instead.

The reference ligand joins the library as well. It was docked already in section 4, but from its own
bound shape, an advantage none of the analogues get: on CpABC1 that is -10.9 kcal/mol against -7.1
for the same molecule rebuilt from a SMILES string. Docking it again the same way as the rest gives
a number the analogues can fairly be compared with.

In [ ]:
LIBRARY_SIZE = 10

urllib.request.urlretrieve("https://raw.githubusercontent.com/ersilia-os/ub-cedd-projects-workshop/main/sandbox/silymarin_analogues.csv", f"{WORK}/silymarin_analogues.csv")
library = pd.read_csv(f"{WORK}/silymarin_analogues.csv").head(LIBRARY_SIZE)

reference_row = pd.DataFrame([{"name": "reference", "smiles": Chem.MolToSmiles(Chem.RemoveHs(ligand))}])
library = pd.concat([reference_row, library], ignore_index=True)
library[["name", "smiles"]]

A SMILES string carries no coordinates, so RDKit builds one 3D shape per compound and relaxes it
with a force field. One shape each is the cheap choice; a real campaign would generate several, since
gnina only explores the rotatable bonds from the shape it is given.

In [ ]:
from rdkit.Chem import AllChem

def embed(smiles, name):
    """Turn a SMILES string into a single relaxed 3D structure."""
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    AllChem.EmbedMolecule(mol, randomSeed=42)
    AllChem.MMFFOptimizeMolecule(mol)
    mol.SetProp("_Name", name)
    return mol

molecules = [embed(s, n) for n, s in zip(library["name"], library["smiles"])]
print(f"{len(molecules)} molecules with 3D coordinates")

## 7. Dock the library

Every compound goes through the same command, into the same box. Big flexible molecules take around
210 seconds and small rigid ones around 110, so eleven compounds take roughly half an hour. Each
result prints as it arrives.

> **Note:** keep the browser tab awake. Colab buffers output while the tab is disconnected, so a
> sleeping laptop makes the run look stuck when it is not.

In [ ]:
import time

rows = []
for i, mol in enumerate(molecules, start=1):
    name = mol.GetProp("_Name")
    tag = "lib_" + name.replace(" ", "_")
    Chem.MolToMolFile(mol, f"{WORK}/{tag}.sdf")
    start = time.time()
    best = read_scores(dock(f"{WORK}/{tag}.sdf", tag)).sort_values("affinity").iloc[0]
    rows.append({"name": name, "affinity": best.affinity,
                 "cnn_pose_score": best.cnn_pose_score, "cnn_affinity": best.cnn_affinity})
    print(f"{i:2d}/{len(molecules)}  {name:<20} affinity {best.affinity:6.2f}  ({time.time() - start:.0f} s)")

results = pd.DataFrame(rows)

## 8. Rank the compounds

Ranked by Vina affinity, most negative first, against the reference ligand docked the same way.

> **Note:** these are predictions on a computed receptor, from one conformer per compound and one
> docking run each. They are a way to order compounds for a closer look, nothing more.

In [ ]:
reference_affinity = results.loc[results["name"] == "reference", "affinity"].iloc[0]

ranked = results.sort_values("affinity").reset_index(drop=True)
ranked["beats_reference"] = ranked["affinity"] < reference_affinity
ranked.to_csv(f"{WORK}/docking_scores.csv", index=False)
print(f"reference: {reference_affinity:.2f} | compounds that beat it: {int(ranked['beats_reference'].sum())}")
ranked

The same numbers as a plot, with the reference drawn as a horizontal line. The bars point downwards
because a binding energy is negative, so the ones reaching furthest down are the best.

In [ ]:
import stylia

stylia.set_format("slide")
stylia.set_style("ersilia")
colors = stylia.NamedColors()

fig, axs = stylia.create_figure(1, 1)
ax = axs.next()
ax.bar(ranked["name"], ranked["affinity"],
       color=[colors.plum if b else colors.gray for b in ranked["beats_reference"]])
ax.axhline(reference_affinity, color=colors.blue)
ax.tick_params(axis="x", rotation=90)
stylia.label(ax, xlabel="", ylabel="Affinity (kcal/mol)")
stylia.save_figure(f"{WORK}/docking_scores.png")

Finally the best analogue in the pocket, with the uploaded ligand behind it in grey.

In [ ]:
best = ranked[ranked["name"] != "reference"].iloc[0]
top_pose = best_pose(f"{WORK}/lib_{best['name'].replace(' ', '_')}_docked.sdf")
Chem.MolToMolFile(top_pose, f"{WORK}/top_hit.sdf")
print(f"Best analogue: {best['name']} (affinity {best['affinity']:.2f} kcal/mol)")
view_pose(f"{WORK}/top_hit.sdf", f"{WORK}/ligand.sdf")

## Summary

- If section 4 ran and section 5 printed a small RMSD for the Vina-best pose, gnina works in Colab
  and the pocket is set up correctly. If gnina would not start, the CUDA 12 libraries in section 1
  are the first place to look.
- Section 5 is also the useful lesson: on a computed receptor the CNN ranking put the correct pose
  fifth, so redocking is what tells you which score to trust before ranking anything.
- Results are in `work/docking_scores.csv` and `work/docking_scores.png`, both wiped with the
  session.

Rough edges worth knowing:

- The binary is 1.4 GB and rebuilt against CUDA 12, so it needs the library shim in section 1 on
  today's Colab image.
- The search runs on the CPU and Colab gives two cores, so the GPU only speeds up the CNN rescoring.
  Half an hour for eleven compounds is the realistic cost.
- No protonation states, one conformer per compound, no receptor flexibility.

**Next:** if this earns a place in the blue group's project, the version for participants would read
the complex from `projects/blue/data/` instead of an upload, and the screen would need to be short
enough to sit inside a workshop session.